# meas_crowd sample run

One cutout plus a small detector region, with every plot of the user guide
(`doc/meas_crowd_pipeline_guide.docx`).  Run the cells in order with the LSST stack kernel.

A Jupyter kernel does not see `setup -k -r .` from a terminal, so the first code cell puts the
meas_crowd checkout on the path itself: set `MEAS_CROWD` there if the notebook is not in the
checkout's `examples/` folder.

In [ ]:
%matplotlib inline
import os
import sys

# where the meas_crowd checkout is (default: $MEAS_CROWD_DIR from eups, else the folder above
# this notebook, i.e. the checkout when the notebook sits in examples/)
MEAS_CROWD = os.environ.get("MEAS_CROWD_DIR") or os.path.abspath("..")
# MEAS_CROWD = "/sdf/data/rubin/user/abudlong/code/build/meas_crowd"
PKG = os.path.join(MEAS_CROWD, "python")
if not os.path.isdir(os.path.join(PKG, "lsst", "meas", "crowd")):
    raise FileNotFoundError(f"no meas_crowd package under {PKG}: set MEAS_CROWD above")
if PKG not in sys.path:
    sys.path.insert(0, PKG)
import lsst  # noqa: E402
import lsst.meas  # noqa: E402
for mod, sub in ((lsst, "lsst"), (lsst.meas, os.path.join("lsst", "meas"))):
    if os.path.join(PKG, sub) not in list(mod.__path__):   # lsst / lsst.meas were imported before
        mod.__path__.append(os.path.join(PKG, sub))

import glob  # noqa: E402
import time  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402
from lsst.daf.butler import Butler  # noqa: E402

import lsst.meas.crowd as crowd  # noqa: E402

print("meas_crowd from:", os.path.dirname(crowd.__file__))
print("meas_crowd driver:", crowd.pipeline.__version__, "| C++ kernels:", crowd.HAVE_CXX)

## Settings
`PSF_SOURCE = "visit"` uses the visit image's own PSF; `"refit"` builds a new detector-level PSF
(slower; also gives a detector-level background and PSF-star list).

In [ ]:
VISIT, DETECTOR = 2025062100433, 144
CUTOUT = "1000,1000,300"            # x0, y0, size (detector px)
PAD = 20
PSF_SOURCE = "visit"                # or "refit"
REPO, COLLECTION = "dp2_prep", "LSSTCam/runs/DRP/DP2"
REGION = (800, 1599, 800, 1599)     # detector region for the tile run: 2 x 2 tiles of 400 px
OUT = "sample_nb"
os.makedirs(OUT, exist_ok=True)

## 1. Run the pipeline on one cutout

In [ ]:
butler = Butler(REPO, collections=COLLECTION)
refs = butler.query_datasets("visit_image", collections="LSSTCam/runs/*", find_first=False,
                             where=f"instrument='LSSTCam' AND visit={VISIT} AND detector={DETECTOR}")
visitImage = butler.get(refs[0])
dataId = refs[0].dataId

cfg = crowd.makeConfig(crowd.CROWDED_FIELD_CONFIG)
sess = crowd.prepareSession(VISIT, DETECTOR, [CUTOUT], pad=PAD, psfSource=PSF_SOURCE,
                             visitImage=visitImage)
crowd.attachGaia(sess, visitImage=visitImage, repo=REPO)      # cached in gaia_cache/

t0 = time.perf_counter()
runs = crowd.runSessionGaia(sess, cfg, outDir=OUT, maxOuter=3, maxInner=15, plotFinal=False)
if not runs:
    raise RuntimeError("the cutout fit failed -- see the traceback printed above")
name, run = next(iter(runs.items()))
print(f"fit: {time.perf_counter() - t0:.0f} s")
crowd.report(run)

In [ ]:
crowd.plotFinalComparison(run)
plt.show()
crowd.plotResidualGray(run)
plt.show()
phot = run.photometry          # made by the fit, with the Gaia comparison columns;
                               # saved as {OUT}/visit*_det*/<cutout>/photometry.fits
phot[:10]

### Comparison with Gaia
`gaiaMatch`, the position offsets (`gaiaDx/dy` [px], `gaiaDra/ddec` [mas]) and the magnitude
difference (`gaiaDmag`, and `gaiaDmagRel` with the median offset removed) are columns of
`phot`; `doc/catalog_columns.md` describes every column.

In [ ]:
crowd.plotGaiaComparison(phot)
plt.show()
phot["id", "psfMag", "gaiaMatch", "gaiaMatchType", "gaiaG", "gaiaDmagRel", "gaiaDx", "gaiaDy"][:10]

## 2. PSF candidate stars and the PSF model
Orange circles = candidates, green squares = stars used.  The visit PSF's stars come from a source
catalogue with `calib_psf_*` flags; its dataset name depends on the pipeline version
(`butler.registry.queryDatasetTypes("*star*")` lists them).

In [ ]:
if PSF_SOURCE == "refit":
    stars = crowd.psfStarPositions(session=sess)
else:
    cat = None
    for ds in ("single_visit_star_unstandardized", "single_visit_star_footprints", "src"):
        try:
            cat = butler.get(ds, dataId=dataId, collections="LSSTCam/runs/*")
            print("PSF stars from", ds)
            break
        except Exception as e:
            print(f"  {ds}: {type(e).__name__}")
    stars = crowd.psfStarPositions(catalog=cat) if cat is not None else None
crowd.showPsfStars(visitImage, stars, psf=sess.psf)
plt.show()

In [ ]:
# the PSF update inside the cutout after each outer pass (residual stack of clean stars)
hist = run.outerHistory
for k in range(len(hist) - 1):
    crowd.showPsfStars(hist[k]["exposure"], crowd.psfStarPositions(run=run, outer=k),
                         psf=hist[k + 1]["psf"])
    plt.show()
crowd.fitLib.plotPsfModelsByOuterPass(hist)

## 3. Initial background and the background of every pass

In [ ]:
crowd.showBackgrounds(run, initial=sess.bgImage)
plt.show()

## 4. Model images and residuals per round

In [ ]:
crowd.showRounds(run, level="outer")
plt.show()
for k in range(len(hist)):
    crowd.showRounds(run, level="inner", outer=k, maxRows=6)
    plt.show()
crowd.fitLib.plotOuterLoopSummary(hist)

### Which sources were lost or gained between passes, and why
For each lost source: is its flux still in the next pass's residual (a real loss), was it merged
into a neighbour, removed by the halo / ring check (blocked from re-detection), or not needed
(likely spurious)?

In [ ]:
for k in range(len(hist) - 1):
    changes = crowd.comparePasses(run, a=k, b=k + 1)
    plt.show()
changes

## 5. Source cutouts linked to boxes on the data / model / residual
Each coloured box on the overview is exactly the window of that source's row on the pages,
framed in the same colour.  Pass `sources=[index or (x, y), ...]` to choose sources.

In [ ]:
crowd.showLinkedSourcePages(run, sourceView="all", sourcesPerPage=6, maxPages=1)
plt.show()

In [ ]:
# every source: one PNG per page (an overview with that page's boxes + the page), written and
# freed one at a time -- a single PDF of all pages keeps every image in memory until it closes
pages = crowd.showLinkedSourcePages(run, maxPages=None, outDir=f"{OUT}/{name}_source_pages")
# the magnitude-labelled cutouts of the worst-fitting sources
crowd.plotSourceCutouts(run, how="worst", n=8, phot=phot)

## 6. Individual tiles of a detector region
Four 400 px tiles, each fitted with 30 px padding (takes a while: about four cutout fits).
Set `region=None` for the whole detector (about 100 tiles -- better as the Slurm array,
`examples/usdf_detector_array.sh`).

In [ ]:
detDir = f"{OUT}/detector_run"
crowd.fitTiles(VISIT, DETECTOR, cfg, outDir=detDir, tile=400, pad=30, region=REGION,
                repo=REPO, collection=COLLECTION, psfSource=PSF_SOURCE,
                visitImage=visitImage if PSF_SOURCE == "visit" else None)
for path in sorted(glob.glob(f"{detDir}/tile_*.fits")):
    crowd.plotTile(path)

## 7. Stitched data, model and residual

In [ ]:
mosaic = crowd.stitchTiles(detDir)
crowd.plotDetector(mosaic)
x0, x1, y0, y1 = REGION
crowd.plotDetector(mosaic, view=(x0, x0 + 199, y0, y0 + 199), binning=1, sources=True)